In [1]:
from google.colab import drive


# Montar Google Drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
!pip install pyomo
from pyomo.environ import *
import matplotlib.pyplot as plt
!wget -N -q "https://matematica.unipv.it/gualandi/solvers/ipopt-linux64.zip"
!unzip -o -q ipopt-linux64

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 1.6 MB/s eta 0:00:00


In [3]:
!pip install pyproj

Steps for the Model:
Inputs:

Forecast precipitation (for the next 1–3 days).
Last measured precipitation.
Temperature.
River flow.
Water level.
Constraints:

Maximum frequency: once every 10 minutes.
Minimum frequency: once a day (1440 minutes).
Behavior:

Increase frequency (reduce time between queries) if:
Forecast precipitation is higher than the last measured precipitation.
Measured precipitation has increased.
River flow exceeds 112.
Water level has increased.
Decrease frequency if the temperature has increased.
Objective: Find the optimal query frequency based on these factors.

Formulation:
We can model this problem with a linear relationship between the inputs and the frequency output. Heres a simple formulation:


Where:

Base frequency could start as, for example, 1440 minutes (once a day).
\alpha_1, \alpha_2, \alpha_3, \alpha_4 are parameters to adjust how sensitive the frequency is to each input.

Sure! Here’s the formulation in LaTeX that captures the objective and constraints of the problem:

### Objective Function:
Minimize the query frequency based on environmental conditions:

$[\min_{f} \left( f - \alpha_1 \cdot \max(0, \bar{p}_{\text{forecast}} - p_{\text{last}}) - \alpha_2 \cdot \max(0, p_{\text{measured}} - p_{\text{last}}) + \alpha_3 \cdot \max(0, T_{\text{current}} - T_{\text{last}}) - \alpha_4 \cdot \max(0, L_{\text{current}} - L_{\text{last}}) \right)]$

Where:
- \( f \) is the query frequency (in minutes).
- \( $\bar{p}_{\text{forecast}}$ \) is the average forecasted precipitation over the next days.
- \( $p_{\text{last}}$ \) is the last measured precipitation.
- \( $p_{\text{measured}}$ \) is the current measured precipitation.
- \( $T_{\text{current}}$ \) and \( $T_{\text{last}}$ \) are the current and last measured temperatures.
- \( $L_{\text{current}}$ \) and \( $L_{\text{last}}$ \) are the current and last measured water levels.
- \( $\alpha_1, \alpha_2, \alpha_3, \alpha_4$ \) are weighting parameters adjusting the sensitivity.

### Constraints:

1. **Bounds on frequency**:
   \[
   $f_{\text{min}}$ \leq f \leq f_{\text{max}}
   \]
   Where:
   - \( $f_{\text{min}}$ = 10 \) minutes (maximum frequency).
   - \( $f_{\text{max}}$ = 1440 \) minutes (minimum frequency, once a day).

2. **River flow constraint**:
   \[
   $f = f_{\text{min}} \quad \text{if} \quad F_{\text{current}} > F_{\text{threshold}}$
   \]
   Where \( $F_{\text{current}}$ \) is the current river flow and \( $F_{\text{threshold}} = 112$ \).

This is the basic structure of your model. You can tweak the parameters \( $\alpha_1, \alpha_2, \alpha_3, \alpha_4$ \) based on the importance of each factor to adjust the frequency more or less aggressively.

In [4]:
from pyomo.environ import *

# Inputs (example data)
forecast_precipitation = [11.4, 16.2, 20.1]  # Forecasted precipitation for the next 3 days
last_measured_precipitation = 10.0  # Last measured precipitation
current_precipitation = 11.2  # Current measured precipitation
last_temperature = 10  # Last measured temperature
current_temperature = 11  # Current measured temperature
river_flow = 111.2  # Current river flow
water_level = 14.5  # Current water level
last_water_level = 13.2  # Last measured water level

# Parameters
max_frequency = 10  # Maximum frequency in minutes (every 10 minutes)--------------- This variable will also limit the maximum number of estimations in the other code as a constraint. It will depend on how much time an estimation will take (experimental)
min_frequency = 1440  # Minimum frequency in minutes (once a day)
river_flow_threshold = 112  # If river flow exceeds this value, set to max frequency

# Create the model
model = ConcreteModel()

# Variable: Frequency in minutes
model.frequency = Var(bounds=(max_frequency, min_frequency), within=NonNegativeReals)

# Objective: Maximize the safety (i.e., minimize the frequency to query more often when needed)
def objective_rule(model):
    avg_forecast_precipitation = sum(forecast_precipitation) / len(forecast_precipitation)

    # Conditions that affect the frequency
    forecast_increase = max(0, avg_forecast_precipitation - last_measured_precipitation)
    measured_precipitation_increase = max(0, current_precipitation - last_measured_precipitation)
    temperature_increase = max(0, current_temperature - last_temperature)
    water_level_increase = max(0, water_level - last_water_level)

    # Base frequency (1440 minutes), so we can adjust alphas
    return (
        model.frequency
        - 100 * forecast_increase  # Increase frequency if forecast precipitation rises
        - 200 * measured_precipitation_increase  # Increase if measured precipitation rises
        + 300 * temperature_increase  # Decrease frequency if temperature increases
        - 150 * water_level_increase  # Increase if water level rises
    )

# Set objective to minimize frequency (minimize model.frequency)
model.objective = Objective(rule=objective_rule, sense=minimize)

# Constraint: If river flow exceeds threshold, set to max frequency
def river_flow_constraint(model):
    if river_flow > river_flow_threshold:
        return model.frequency == max_frequency
    return Constraint.Skip

model.river_flow_constr = Constraint(rule=river_flow_constraint)

# Solver
solver = SolverFactory('ipopt')
result = solver.solve(model)

# Output the optimal frequency in minutes
print(f"Optimal query frequency: {model.frequency()} minutes")


Optimal query frequency: 10.0 minutes


In [4]:
from pyomo.environ import *

# Inputs (example data)
forecast_precipitation = [11.4, 16.2, 20.1]  # Forecasted precipitation for the next 3 days
last_measured_precipitation = 10.0  # Last measured precipitation
current_precipitation = 11.2  # Current measured precipitation
last_temperature = 10  # Last measured temperature
current_temperature = 11  # Current measured temperature
river_flow = 111.2  # Current river flow
water_level = 14.5  # Current water level
last_water_level = 13.2  # Last measured water level

# Parámetros del problema
N = 2  # Número de nodos
battery_levels = [15.7, 12.1]  # Último nivel de las baterías (en Voltios)
min_battery_voltage = 10.0  # Voltaje mínimo requerido para que los nodos operen
estimation_cost = 0.5  # Voltaje consumido por cada estimación realizada
estimation_time = 1  # Tiempo que toma realizar una estimación (en minutos)
max_frequency = 10  # Frecuencia máxima (cada 10 minutos)

# Crear el modelo
model = ConcreteModel()

# Variables de decisión: número de estimaciones a realizar en cada nodo
model.estimations = Var(range(N), within=NonNegativeIntegers)

# Función objetivo: Maximizar la cantidad de estimaciones, ajustada por condiciones climáticas
def objective_rule(model):
    avg_forecast_precipitation = sum(forecast_precipitation) / len(forecast_precipitation)

    # Condiciones que afectan el número de estimaciones
    forecast_increase = max(0, avg_forecast_precipitation - last_measured_precipitation)
    measured_precipitation_increase = max(0, current_precipitation - last_measured_precipitation)
    temperature_increase = max(0, current_temperature - last_temperature)
    water_level_increase = max(0, water_level - last_water_level)

    # Maximizar las estimaciones, ajustado por los factores externos
    return (
        sum(model.estimations[i] for i in range(N))
        + 100 * forecast_increase  # Aumentar estimaciones si aumenta la precipitación pronosticada
        + 200 * measured_precipitation_increase  # Aumentar si aumenta la precipitación medida
        - 100 * temperature_increase  # Disminuir estimaciones si aumenta la temperatura
        + 150 * water_level_increase  # Aumentar si sube el nivel del agua
    )

# Establecer la función objetivo
model.objective = Objective(rule=objective_rule, sense=maximize)

# Restricción de batería: el nivel de voltaje de cada batería debe ser suficiente para realizar las estimaciones
def battery_constraint(model, i):
    return battery_levels[i] - model.estimations[i] * estimation_cost >= min_battery_voltage

model.battery_constraints = Constraint(range(N), rule=battery_constraint)

# Restricción de tiempo: las estimaciones no deben exceder la frecuencia máxima (cada 10 minutos)
def time_constraint(model, i):
    return model.estimations[i] * estimation_time <= max_frequency

model.time_constraints = Constraint(range(N), rule=time_constraint)

# Restricción adicional: si el flujo del río excede el umbral, se establece la frecuencia máxima
river_flow_threshold = 112  # Umbral de caudal del río
def river_flow_constraint(model):
    if river_flow > river_flow_threshold:
        return sum(model.estimations[i] for i in range(N)) == max_frequency // estimation_time
    return Constraint.Skip

model.river_flow_constr = Constraint(rule=river_flow_constraint)

# Crear un solver
solver = SolverFactory('ipopt')

# Resolver el problema
result = solver.solve(model)

# Obtener el resultado como un array de estimaciones
output = [int(model.estimations[i]()) for i in range(N)]

# Mostrar el array de resultados
print("Número óptimo de estimaciones por nodo:", output)


Número óptimo de estimaciones por nodo: [10, 4]


Explicación del código

1.   Función objetivo

  El objetivo del modelo es maximizar el número de estimaciones que se pueden realizar en cada nodo, teniendo en cuenta los efectos de las condiciones ambientales (precipitación, temperatura, nivel del agua) y con el propósito de ajustar dinámicamente la frecuencia de las estimaciones en función de estas variables.
2.   Parámetros de frecuencia
Maximización del número total de estimaciones de los nodos, sumado a:
- Aumentar estimaciones si sube la precipitación pronosticada: 100 * forecast_increase
- Aumentar estimaciones si sube la precipitación medida: 200 * measured_precipitation_increase
- Disminuir estimaciones si sube la temperatura: -300 * temperature_increase
- Aumentar estimaciones si sube el nivel del agua: 150 * water_level_increase

**Importante** Sedebe considerar que los alphas o costos asociados a cada variable se deben modificar según lo que veamos conveniente.

3. Constraints

- Nivel de batería: El voltaje restante de cada nodo después de las estimaciones debe ser mayor o igual al voltaje mínimo permitido.
 - Nivel de batería restante: battery_levels[i] - model.estimations[i] * estimation_cost
 - Voltaje mínimo: min_battery_voltage = 10.0 V
 - Coste por estimación: estimation_cost = 1.5 V

- Time constraint: El número de estimaciones está limitado por la frecuencia máxima, que en este caso es cada 10 minutos.
  - Frecuencia máxima: max_frequency = 10 minutos
  - Tiempo por estimación: estimation_time = 2 minutos

**Importante** La frecuencia máxima y el tiempo de estimación son valores que deberán medirse experimentalmente una vez implementados.
- River flow constraint: Si el caudal del río supera un umbral predeterminado, el sistema fuerza la máxima frecuencia posible de estimaciones.
  - Umbral del caudal del río: river_flow_threshold = 112 m³/s
  - Si el caudal excede este valor, el número de estimaciones será el máximo posible bajo la restricción de tiempo.

  5. Datos de entrada:
- Precipitación pronosticada: forecast_precipitation = [11.4, 16.2, 20.1] (mm)
- Precipitación medida: last_measured_precipitation = 10.0 y current_precipitation = 11.2 (mm)
- Temperatura medida: last_temperature = 10, current_temperature = 11 (°C)
- Caudal del río: river_flow = 111.2 (m³/s)
- Nivel del agua: last_water_level = 13.2, current_water_level = 14.5 (m)
- Niveles de batería: battery_levels = [10.7, 10.1] (V)


